# 🌾 Imarika Agricultural AI - QLoRA Fine-tuning with CSV Data

Fine-tune Llama 3.2 3B using your actual CSV datasets.

**Crops**: Beans, Cassava, Finger Millet, Maize, Sorghum, Sweet Potatoes

In [1]:
# Install dependencies
!pip install -q torch transformers peft datasets accelerate bitsandbytes trl
!nvidia-smi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 38.7 MB/s eta 0:00:00
Fri Oct 24 14:43:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |    

In [2]:
import torch
import json
import random
import pandas as pd
from google.colab import files
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA Available: True
GPU: Tesla T4


In [3]:
# Upload CSV datasets
print("Upload your CSV files (beans.csv, cassava.csv, etc.)")
uploaded = files.upload()

# Load CSV data
crop_data = {}
for filename in uploaded.keys():
    if filename.endswith('.csv'):
        crop_name = filename.replace('.csv', '')
        df = pd.read_csv(filename)
        crop_data[crop_name] = df
        print(f"Loaded {crop_name}: {len(df)} records")
        print(f"Columns: {list(df.columns)}")

print(f"\nTotal crops loaded: {len(crop_data)}")

Upload your CSV files (beans.csv, cassava.csv, etc.)


Saving finger_millet.csv to finger_millet.csv
Saving fao_temp.csv to fao_temp.csv
Saving maize.csv to maize.csv
Saving cassava.csv to cassava.csv
Saving beans.csv to beans.csv
Saving sweet_potatoes.csv to sweet_potatoes.csv
Saving fao_cc.csv to fao_cc.csv
Saving sorghum.csv to sorghum.csv
Loaded finger_millet: 5 records
Columns: ['Growth Stages', 'Farming Activities', 'Normal Conditions (ideal condition)', 'Wetter than Normal (Higher than ideal)', 'Drier than Normal (Lower than Ideal)']
Loaded fao_temp: 38 records
Columns: ['Crop', 'Altitude_Range', 'Soil_Type(pH)', 'Min Temp', 'Max Temp', 'Min Rainfall', 'Min_Rainfall', 'Max_Rainfall']
Loaded maize: 5 records
Columns: ['Growth Stages', 'Farming Activities', 'Normal Conditions (ideal condition)', 'Wetter than Normal (Higher than ideal)', 'Drier than Normal (Lower than Ideal)', 'Normal Conditions (ideal condition)-English', 'Swahili']
Loaded cassava: 5 records
Columns: ['Growth Stages', 'Farming Activities', 'Normal Conditions (ideal co

In [4]:
# Generate training data from CSV
def generate_from_csv(crop_data, num_samples=1500):
    dataset = []

    templates = [
        "What nutrients does {crop} need?",
        "How do I manage pests in {crop}?",
        "What weather conditions are best for {crop}?",
        "When should I plant {crop}?",
        "How do I harvest {crop}?",
        "What are the growth stages of {crop}?",
        "What fertilizers work best for {crop}?",
        "How do I prepare soil for {crop}?"
    ]

    for _ in range(num_samples):
        crop = random.choice(list(crop_data.keys()))
        question = random.choice(templates).format(crop=crop)

        # Extract info from CSV
        df = crop_data[crop]
        if not df.empty:
            row = df.sample(1).iloc[0]

            # Create response from CSV data
            response_parts = []
            for col in df.columns:
                if pd.notna(row[col]) and str(row[col]).strip():
                    response_parts.append(f"{col}: {row[col]}")

            if response_parts:
                response = f"For {crop}: " + "; ".join(response_parts[:3])
            else:
                response = f"For {crop}, follow standard East African farming practices."
        else:
            response = f"For {crop}, follow standard East African farming practices."

        conversation = {
            "messages": [
                {"role": "system", "content": "You are Imarika, an expert agricultural advisor specializing in East African crops: beans, cassava, finger millet, maize, sorghum, and sweet potatoes. Provide detailed, practical farming advice."},
                {"role": "user", "content": question},
                {"role": "assistant", "content": response}
            ]
        }
        dataset.append(conversation)

    return dataset

# Generate training dataset
training_data = generate_from_csv(crop_data, 1500)
print(f"Generated {len(training_data)} training samples from CSV data")

# Show sample
print("\nSample conversation:")
print(json.dumps(training_data[0], indent=2))

Generated 1500 training samples from CSV data

Sample conversation:
{
  "messages": [
    {
      "role": "system",
      "content": "You are Imarika, an expert agricultural advisor specializing in East African crops: beans, cassava, finger millet, maize, sorghum, and sweet potatoes. Provide detailed, practical farming advice."
    },
    {
      "role": "user",
      "content": "How do I manage pests in sweet_potatoes?"
    },
    {
      "role": "assistant",
      "content": "For sweet_potatoes: Growth Stages: Planting ; Farming Activities: Sowing; Normal Conditions (ideal condition): Plant early at appropriate spacing and where intercropping is\n done with maize, then spacing for maize is adjusted to accommodate the sweet potato\nPlant vines at an angle of 45 degrees, 30 cm apart, \nwith vine ends (bases) towards the centre of the ridge, \u2154 covered with soil, leaving \u2153 above the soil. \nWhere ridges are wider than 1.0 m, double rows at 30 cm apart can be planted on ridge.\n

In [5]:
# Model configuration
model_name = "meta-llama/Llama-3.2-3B-Instruct"

# QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# LoRA config
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)

print("QLoRA configuration ready")

QLoRA configuration ready


In [7]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config) # Use the previously defined bnb_config

messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


I'm an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."<|eot_id|>


In [8]:
# Prepare dataset
def format_chat(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(training_data)
dataset = dataset.map(format_chat)

print(f"Dataset prepared: {len(dataset)} samples")
print(f"Sample text length: {len(dataset[0]['text'])} characters")

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Dataset prepared: 1500 samples
Sample text length: 1347 characters


In [9]:
# Training arguments
training_args = TrainingArguments(
    output_dir="./imarika-csv-results",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=25,
    save_steps=100,
    optim="paged_adamw_32bit",
    warmup_steps=50,
    max_grad_norm=0.3
)

# Apply PEFT config to the model before initializing the trainer
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
)

print("Trainer initialized - Ready for training!")

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


Tokenizing train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1500 [00:00<?, ? examples/s]

Trainer initialized - Ready for training!


In [10]:
# Start training
print("🔥 Starting QLoRA fine-tuning with CSV data...")
print("This will take 2-3 hours")

trainer.train()

print("✅ Training completed!")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


🔥 Starting QLoRA fine-tuning with CSV data...
This will take 2-3 hours


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kevinobote (kevinobote-strathmore-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
25,2.883800
50,1.365800
75,0.504600
100,0.124800
125,0.069400
150,0.056000
175,0.057800
200,0.044600
225,0.040000
250,0.038700


✅ Training completed!


In [11]:
# Save model
output_dir = "./imarika-csv-agricultural-model"
trainer.model.save_pretrained(output_dir)
trainer.tokenizer.save_pretrained(output_dir)

print(f"Model saved to {output_dir}")
!ls -la {output_dir}/

Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.


Model saved to ./imarika-csv-agricultural-model
total 111916
drwxr-xr-x 2 root root     4096 Oct 24 15:17 .
drwxr-xr-x 1 root root     4096 Oct 24 15:17 ..
-rw-r--r-- 1 root root      943 Oct 24 15:17 adapter_config.json
-rw-r--r-- 1 root root 97307544 Oct 24 15:17 adapter_model.safetensors
-rw-r--r-- 1 root root     3827 Oct 24 15:17 chat_template.jinja
-rw-r--r-- 1 root root     5230 Oct 24 15:17 README.md
-rw-r--r-- 1 root root      296 Oct 24 15:17 special_tokens_map.json
-rw-r--r-- 1 root root    50525 Oct 24 15:17 tokenizer_config.json
-rw-r--r-- 1 root root 17209920 Oct 24 15:17 tokenizer.json


In [12]:
# Test the fine-tuned model
def test_model(query):
    messages = [
        {"role": "system", "content": "You are Imarika, an agricultural expert for East African crops."},
        {"role": "user", "content": query}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )

    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()

# Test with queries based on your CSV data
test_queries = [
    "What nutrients does maize need?",
    "How do I manage pests in beans?",
    "What weather is best for cassava?",
    "When should I plant sorghum?",
    "How do I harvest sweet potatoes?"
]

print("🧪 Testing CSV-trained model:")
print("=" * 50)

for query in test_queries:
    print(f"\nQ: {query}")
    try:
        response = test_model(query)
        print(f"A: {response}")
    except Exception as e:
        print(f"Error: {e}")
    print("-" * 30)

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


🧪 Testing CSV-trained model:

Q: What nutrients does maize need?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


A: For maize: Growth Stages: Planting ; Farming Activities: Nutrient Management ; Normal Conditions (ideal condition): Maize requires three primary nutrients – nitrogen (N), phosphorous (P) and potassium (K).
Potash (K) is the nutrient required in the greatest amount by maize,
Maize is not very sensitive to trace element deficiencies, but boron, copper, zinc, manganese and iron may occasionally be deficient on soils where manure is not applied regularly.
------------------------------

Q: How do I manage pests in beans?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


A: For beans: Growth Stages: Harvest; Farming Activities: Harvesting ; Normal Conditions (ideal condition): Bean will produce up to 1.0–2.0 tons per ha
Dry pods in the sun, thresh and winnow to remove chaff and dust, re-dry to attain storage moisture content level.
Dry the pods on raised platforms, plastic, mat to avoid contact with moisture and other impurities.
It is advisable to do the threshing when the moisture level is 14-15%, If the seeds are too dry, the pods can be easily damaged during threshing, thresh the pods manually by beating with a stick.
After winnowing, grade the seeds to remove dust, damaged, broken and diseased seeds, separate the seeds by variety
------------------------------

Q: What weather is best for cassava?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


A: For cassava: Growth Stages: Planting ; Farming Activities: Land Preparation ; Normal Conditions (ideal condition): Cassava should be grown away from trees to avoid shading Ensure deep and light soil to allow root development
Cassava mostly utilizes nitrogen fixed by leguminous plants
Prepare land early, especially during the dry season, preferably 2 months before planting.
Harrow to a medium tilth.
Analyse the soil and cassava nutrient requirement for various soil types. 
Add manure and fertilizers to the soil in the right amounts to provide the required plant.
------------------------------

Q: When should I plant sorghum?


Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


A: For sorghum: Groth Stages: Planting ; Farming Activities: Nutrient Management ; Normal Conditions (ideal condition): nutrient requirements include: N, P, K, Zn, S, Cl, Mg,
 Ca, Fe, B, Cu, Mn
------------------------------

Q: How do I harvest sweet potatoes?
A: For sweet_potatoes: Growth Stages: Harvest; Farming Activities: Post Harvest Handling ; Normal Conditions (ideal condition): Keep harvested roots protected from direct sunshine to avoid shrinkage and shrivelling, 
Wash, sort, and grade the harvested roots, depending on the intended use
Washing is to remove soil from the roots without bruising the skin
Sorting is done to remove rotten, damaged, malformed and weevil infested roots, and debris
Grading is done to sorted roots to have uniform produce, based on size, shape, 
colour, weight, maturity, market demand
------------------------------


In [14]:
# Create download package
import zipfile
import os
from google.colab import files

zip_name = "imarika_csv_qlora_model.zip"
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_in_dir in os.walk(output_dir):
        for file in files_in_dir:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_dir)
            zipf.write(file_path, arcname)

print(f"📦 Model packaged: {zip_name}")
print(f"📊 Size: {os.path.getsize(zip_name) / 1e6:.1f}MB")

# Download the model
files.download(zip_name)

print("\n🎉 Your CSV-trained agricultural model is ready!")
print("\n📋 Integration steps:")
print("1. Extract zip to fine_tuning/models/")
print("2. Update ollama_integration.py to load QLoRA model")
print("3. Test with your LangGraph system")

📦 Model packaged: imarika_csv_qlora_model.zip
📊 Size: 92.6MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 Your CSV-trained agricultural model is ready!

📋 Integration steps:
1. Extract zip to fine_tuning/models/
2. Update ollama_integration.py to load QLoRA model
3. Test with your LangGraph system
